In [25]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [27]:
stu_per=pd.read_csv('student-por.csv')

In [29]:
stu_per.head()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,4,0,11,11
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,2,9,11,11
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,6,12,13,12
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,0,14,14,14
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,0,11,13,13


In [31]:
stu_per=stu_per.drop(columns=["school","address","famsize","Pstatus","guardian","nursery","Dalc","Walc","reason"])

In [33]:
stu_per.shape

(649, 24)

In [35]:
X=stu_per.iloc[:,0:-1]
y=stu_per.iloc[:,-1]

In [37]:
print(X.shape)
print(y.shape)

(649, 23)
(649,)


In [39]:
X_train,X_test,y_train,y_test= train_test_split(X,y,test_size=0.2,random_state=42)

In [41]:
numeric_features = [
    'age', 'Medu', 'Fedu', 'traveltime', 'studytime', 
    'failures', 'absences', 'famrel', 'freetime', 'goout', 'health','G1','G2'
]

In [43]:
binary_features = [
    'sex','schoolsup','famsup','paid','activities',
    'higher','internet','romantic'
]

In [45]:
multi_cat_features = ['Mjob', 'Fjob']

In [47]:
# handling outliers and skewness

In [49]:
def transform_absences(X_df):
    X_df=X_df.copy()
    if 'absences' in X_df.columns:
        # Outlier capping using IQR
        Q1= X_df['absences'].quantile(0.25)
        Q3= X_df['absences'].quantile(0.75)
        IQR= Q3-Q1
        upper= Q3 + 1.5 * IQR
        lower= Q1 - 1.5 * IQR
        X_df['absences']= np.clip(X_df['absences'],lower,upper)

        # now log transformation to handle skewness
        X_df['absences']= np.log1p(X_df['absences'])
    return X_df

In [51]:
absences_transformer= FunctionTransformer(transform_absences)

In [53]:
# encoding binary features

In [55]:
def binary_encoder(X_df):
    return X_df.replace({'yes':1, 'no':0, 'F':0, 'M':1})

In [57]:
binary_transformer= FunctionTransformer(binary_encoder)

In [59]:
numeric_pipeline= Pipeline(steps=[
    ('absences_fix',absences_transformer),
    ('imputer',SimpleImputer(strategy='median')),
    ('scaler',StandardScaler())
]
    
)

In [61]:
binary_pipeline= Pipeline(steps=[
    ('encoder',binary_transformer)
]
    
)

In [63]:
multi_cat_pipeline= Pipeline(steps=[
    ('onehot',OneHotEncoder(drop='first',handle_unknown='ignore'))
])

In [65]:
preprocessor= ColumnTransformer(
    transformers=[
        ('num',numeric_pipeline,numeric_features),
        ('bin',binary_pipeline, binary_features),
        ('cat',multi_cat_pipeline, multi_cat_features)
    ]
)

In [67]:
final_pipeline=Pipeline(steps=[
    ('preprocessor',preprocessor)
])

In [69]:
final_pipeline.fit(X_train)

C:\Users\HP\AppData\Local\Temp\ipykernel_8236\212723361.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return X_df.replace({'yes':1, 'no':0, 'F':0, 'M':1})


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('absences_fix',
                                                                   FunctionTransformer(func=<function transform_absences at 0x000001B41D7268E0>)),
                                                                  ('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'Medu', 'Fedu',
                                                   'traveltime', 'studytime',
                                                   'failures', 'absences',
                                                   'famrel', 'freetime',
                                                   'goout', 'health', 'G1',
                                                   'G2']),
                                                 ('bin',
                                                  Pipeline(steps=[('encoder',
                                                                   FunctionTransformer(func=<function binary_encoder at 0x000001B41F54A700>))]),
                                                  ['sex', 'schoolsup', 'famsup',
                                                   'paid', 'activities',
                                                   'higher', 'internet',
                                                   'romantic']),
                                                 ('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore'))]),
                                                  ['Mjob', 'Fjob'])]))])

In [71]:
X_train_ready=final_pipeline.transform(X_train)
X_test_ready=final_pipeline.transform(X_test)

C:\Users\HP\AppData\Local\Temp\ipykernel_8236\212723361.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return X_df.replace({'yes':1, 'no':0, 'F':0, 'M':1})
C:\Users\HP\AppData\Local\Temp\ipykernel_8236\212723361.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return X_df.replace({'yes':1, 'no':0, 'F':0, 'M':1})


In [73]:
print("✅ Preprocessing complete!")
print("X_train shape:", X_train_ready.shape)
print("X_test shape:", X_test_ready.shape)

✅ Preprocessing complete!
X_train shape: (519, 29)
X_test shape: (130, 29)


In [75]:
# Numeric feature names (same as original numeric_features list)
numeric_feature_names = numeric_features

# Binary feature names (same as original binary_features list)
binary_feature_names = binary_features

# Multi-category feature names (after one-hot encoding)
cat_encoder = final_pipeline.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot']
cat_feature_names = cat_encoder.get_feature_names_out(multi_cat_features)

# Combine all
feature_names = np.concatenate([numeric_feature_names, binary_feature_names, cat_feature_names])


In [77]:
X_train_df = pd.DataFrame(X_train_ready, columns=feature_names, index=X_train.index)
X_test_df  = pd.DataFrame(X_test_ready,  columns=feature_names, index=X_test.index)

print(X_train_df.shape)
X_train_df.head()

(519, 29)


,age,Medu,Fedu,traveltime,studytime,failures,absences,famrel,freetime,goout,...,internet,romantic,Mjob_health,Mjob_other,Mjob_services,Mjob_teacher,Fjob_health,Fjob_other,Fjob_services,Fjob_teacher
332,0.987932,-0.437418,-0.256337,-0.754310,1.352962,-0.382133,-1.129611,0.068136,-0.186680,-0.196270,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
29,-0.629534,1.350140,1.591423,-0.754310,0.121054,-0.382133,0.552666,0.068136,0.753970,1.501468,...,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
302,0.987932,0.456361,-0.256337,-0.754310,1.352962,-0.382133,0.018722,1.108214,-0.186680,-1.045139,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
286,0.179199,-0.437418,-1.180216,-0.754310,-1.110853,-0.382133,-1.129611,0.068136,0.753970,-1.045139,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
554,0.179199,-1.331196,-1.180216,0.555011,-1.110853,-0.382133,0.319424,-0.971942,1.694619,1.501468,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [79]:
from sklearn.linear_model import LinearRegression

In [81]:
lr=LinearRegression()
lr.fit(X_train_df,y_train)

LinearRegression()

In [83]:
y_pred=lr.predict(X_test_df)

In [85]:
from sklearn.metrics import r2_score

In [87]:
r2_score(y_test,y_pred)

0.8531091015854222

In [89]:
from sklearn.ensemble import RandomForestRegressor

In [91]:
rf=RandomForestRegressor(max_depth=5)

In [93]:
rf.fit(X_train_df,y_train)

RandomForestRegressor(max_depth=5)

In [95]:
y_pred=rf.predict(X_test_df)

In [97]:
r2_score(y_test,y_pred)

0.8469017257896523

In [99]:
#X_train_df,X_test_df,y_train,y_test

In [101]:
# stacking voting bagging Gradientboosting xgboost SVM


In [103]:
from sklearn.ensemble import GradientBoostingRegressor

In [105]:
from sklearn.ensemble import VotingRegressor

In [107]:
from sklearn.ensemble import BaggingRegressor

In [109]:
from sklearn.svm import SVR

In [143]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor

In [145]:
lr=LinearRegression()
dt=DecisionTreeRegressor()
svr=SVR()
gbr=GradientBoostingRegressor()

In [147]:
estimators=[('LR',lr),('dt',dt),('svr',svr),('gbr',gbr)]

In [149]:
vot=VotingRegressor(estimators=estimators)

In [151]:
vot.fit(X_train_df,y_train)

VotingRegressor(estimators=[('LR', LinearRegression()),
                            ('dt', DecisionTreeRegressor()), ('svr', SVR()),
                            ('gbr', GradientBoostingRegressor())])

In [153]:
y_pred=vot.predict(X_test_df)

In [155]:
r2_score(y_test,y_pred)

0.8513810463861997

In [157]:
from sklearn.model_selection import cross_val_score

In [159]:
for i in estimators:
    scores=cross_val_score(i[1],X_train_df,y_train,cv=5,scoring='r2')
    print(i[0],np.round(np.mean(scores),2))
    

LR 0.84
dt 0.72
svr 0.71
gbr 0.84


In [241]:
from sklearn.neighbors import KNeighborsRegressor
knn=KNeighborsRegressor()

In [301]:
bagging = BaggingRegressor(estimator=None,n_estimators=100,max_samples=0.25,oob_score=True)

In [303]:
bagging.fit(X_train_df,y_train)

BaggingRegressor(max_samples=0.25, n_estimators=100, oob_score=True)

In [304]:
y_pred=bagging.predict(X_test_df)

In [305]:
r2_score(y_test,y_pred)

0.8508240455328754

In [311]:
from sklearn.ensemble import RandomForestRegressor

In [313]:
rf=RandomForestRegressor()

In [315]:
rf.fit(X_train_df,y_train)
y_pred=rf.predict(X_test_df)
r2_score(y_test,y_pred)

0.8382992281740734